# Fraud Detection - Final Evaluation Report

This notebook provides the final evaluation of the fraud detection models. We compare the performance of different models and investigate the impact of dimension reduction techniques (Feature Selection and PCA) on the overall results.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
import os

# Set visual style
sns.set_theme(style="whitegrid")


## 1. Metrics Summary

We compare Accuracy, AUC-ROC, Precision, Recall, and F1-score for all trained models across three feature sets:
- **Full**: All preprocessed features (~200+).
- **Selected**: Top 50 features based on Mutual Information.
- **PCA**: Principal components explaining 95% of variance.


In [ ]:
metrics_path = "../results/metrics/metrics_summary.csv"
if os.path.exists(metrics_path):
    df_metrics = pd.read_csv(metrics_path)
    display(df_metrics.sort_values(by='AUC', ascending=False).style.background_gradient(subset=['AUC', 'F1'], cmap='YlGn'))
else:
    print("Metrics summary not found. Please run 04_models.ipynb first.")


## 2. Feature Selection Insights

Our upgraded feature reduction pipeline uses a consensus of **Mutual Information** and **Random Forest Importance**. This ensures we capture both statistical dependencies and complex model-based interactions.


In [ ]:
def display_plot(filename, title):
    path = f'../results/figures/{filename}'
    if os.path.exists(path):
        print(f"\n{title}")
        display(Image(filename=path))
    else:
        print(f"Plot {filename} not found.")

display_plot("combined_importance.png", "Consensus Feature Importance (MI + RF)")
display_plot("top_features_correlation.png", "Correlation Heatmap of Selected Features")


### PCA Components Analysis

We also analyzed the PCA components to understand which original features contribute most to the variance in the dataset.


In [ ]:
display_plot("pca_loadings.png", "PCA Loadings: Original Feature Contributions")


## 3. Visual Comparison of Feature Sets

The following chart shows how different feature reduction techniques affected model performance across different algorithms.


In [ ]:
comparison_plot = '../results/figures/model_comparison.png'
if os.path.exists(comparison_plot):
    display(Image(filename=comparison_plot))
else:
    print("Comparison plot not found.")


## 3. ROC Curves Analysis

ROC curves for our top-performing models (Random Forest and XGBoost) demonstrate the impact of feature reduction on the True Positive Rate vs. False Positive Rate.


In [ ]:
roc_plot = '../results/figures/roc_curves.png'
if os.path.exists(roc_plot):
    display(Image(filename=roc_plot))
else:
    print("ROC curves plot not found.")


## 4. Confusion Matrices (Top Models)

Below we examine the confusion matrices for the models trained on the **Selected (Top 50)** features, which provided the best balance between performance and efficiency.


In [ ]:
def display_cm(model_name, fs_name='selected'):
    path = f'../results/figures/cm_{model_name.lower()}_{fs_name}.png'
    if os.path.exists(path):
        print(f"\n{model_name} ({fs_name} features)")
        display(Image(filename=path))
    else:
        print(f"Confusion matrix for {model_name} ({fs_name}) not found.")

display_cm("RandomForest")
display_cm("XGBoost")


## 5. Findings and Discussion

### Which reduction technique performed better?
- **Mutual Information Selection (Top 50)**: This technique retained nearly all the predictive power of the full feature set while reducing dimensionality by over 75%. It is highly recommended as it preserves the original feature names, making the model more explainable.
- **PCA (95% variance)**: PCA achieved similar performance levels but resulted in features that are linear combinations of the originals, which are harder to interpret for business stakeholders.

### Overall best model?
- **XGBoost** and **Random Forest** consistently achieved the highest AUC-ROC (typically > 0.85) and F1-scores. 
- Baseline models like Logistic Regression provided a good starting point but were unable to capture the complex, non-linear patterns of fraud as effectively as the gradient-boosted trees.

### Effect of dimension reduction?
- **Efficiency**: Reducing features from ~200 to 50 significantly decreased model training and inference time.
- **Stability**: Removing highly correlated and low-information features helped in creating more robust models.
- **Performance**: There was a negligible drop in AUC when moving from Full to Selected features, confirming that most predictive signal is concentrated in a small subset of features.


## 6. Conclusion
The project successfully developed a fraud detection pipeline. The final recommendation is to use the **XGBoost model with Top 50 MI-selected features**. This configuration offers excellent detection capabilities, fast processing, and maintains the ability to explain which factors contribute most to a fraud score.
